# Week 10 — Ensemble Learning
### Bagging, Boosting, and Voting Classifier Benchmark
**Next Gen Software Hub Pvt. Ltd. — AI & ML Internship**


In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier,
                              VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df = pd.read_csv("datasets/credit_scoring_ensemble.csv")
X = df.drop(columns=["Fraud_Label"])
y = df["Fraud_Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")


## 1. Benchmarking Individual Learners vs Ensembles


In [ ]:
models = {
    "Base: Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Ensemble: Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Ensemble: AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "Ensemble: Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Ensemble: Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42)
}

results = []
for name, clf in models.items():
    start_time = time.time()
    clf.fit(X_train, y_train)
    fit_time = time.time() - start_time
    
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "Train Time (s)": round(fit_time, 4)
    })

res_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
res_df


## 2. Building a Soft Voting Classifier


In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
        ("gb", GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ("lr", LogisticRegression(random_state=42))
    ],
    voting="soft"
)
voting_clf.fit(X_train, y_train)
v_preds = voting_clf.predict(X_test)
v_probs = voting_clf.predict_proba(X_test)[:, 1]

print("--- Soft Voting Classifier Performance ---")
print(f"Voting ROC-AUC: {roc_auc_score(y_test, v_probs):.4f}")
print(f"Voting F1-Score: {f1_score(y_test, v_preds):.4f}")
